In [22]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [23]:
DATA_PATH = "../data/telco_customer_churn.csv"

df = pd.read_csv(DATA_PATH)

In [24]:
new_customer_missing = (
    df["TotalCharges"].str.strip().eq("")
    & df["tenure"].eq(0)
)

df.loc[
    new_customer_missing,
    "TotalCharges"
] = "0"

In [25]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [26]:
X = df.drop(
    columns=["customerID", "Churn"]
)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [28]:
numeric_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [29]:
categorical_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [30]:
scaled_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [31]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [32]:
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            scaled_numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

In [33]:
unscaled_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

In [34]:
unscaled_preprocessor = ColumnTransformer(
    transformers=[
        (
        "numeric",
            unscaled_numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

In [35]:
model_pipelines = {
    "Logistic Regression": Pipeline(
        steps=[
            (
                "preprocessor",
                scaled_preprocessor
            ),
            (
                "model",
                LogisticRegression(
                 max_iter=1000
                )
            )
        ]
    ),

    "KNN": Pipeline(
        steps=[
            (
             "preprocessor",
                scaled_preprocessor
            ),
            (
            "model",
                KNeighborsClassifier()
            )
        ]
    ),

    "SVM-RBF": Pipeline(
        steps=[
            (
             "preprocessor",
                scaled_preprocessor
            ),
            (
            "model",
                SVC(
                    kernel="rbf"
                )
            )
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            (
                "preprocessor",
                unscaled_preprocessor
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=1
                )
            )
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            (
                "preprocessor",
                unscaled_preprocessor
            ),
            (
                "model",
                XGBClassifier(
                    eval_metric="logloss",
                    random_state=42,
                    n_jobs=1,
                    tree_method="hist"
                )
            )
        ]
    )
}

In [36]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [37]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "accuracy": "accuracy"
}

In [38]:
comparison_rows = []

for model_name, model_pipeline in model_pipelines.items():
    results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    comparison_rows.append({
        "Model": model_name,
        "ROC-AUC": results[
            "test_roc_auc"
        ].mean(),
        "ROC-AUC Std": results[
            "test_roc_auc"
        ].std(),
        "PR-AUC": results[
            "test_pr_auc"
        ].mean(),
        "F1": results[
            "test_f1"
        ].mean(),
        "Recall": results[
            "test_recall"
        ].mean(),
        "Precision": results[
            "test_precision"
        ].mean(),
        "Accuracy": results[
            "test_accuracy"
        ].mean(),
        "Ortalama Eğitim Süresi": results[
            "fit_time"
        ].mean()
    })

KeyboardInterrupt: 

In [ ]:
strengths = {
    "Logistic Regression":
     "Hızlı ve güçlü bir baseline modeli.",

    "KNN":
        "Benzer müşterilerin davranışlarından yararlanarak doğrusal olmayan ilişkileri öğrenebilir.",

    "SVM-RBF":
     "RBF kernel sayesinde karmaşık ve eğrisel karar sınırları oluşturabilir.",

    "Random Forest":
     "Özellik etkileşimlerini öğrenir, scaling istemez ve tek karar ağacına göre daha kararlıdır.",

    "XGBoost":
        "Önceki ağaçların hatalarını sırayla düzelterek tabular veride güçlü sonuçlar üretebilir."
}

In [ ]:
weaknesses = {
    "Logistic Regression":
        "Karmaşık doğrusal olmayan ilişkileri ve özellik etkileşimlerini sınırlı öğrenir.",

    "KNN":
        "Scaling ve özellik seçimine duyarlı; yüksek boyutlu veride tahmin süresi uzayabilir.",

    "SVM-RBF":
        "Scaling ile dikkatli C ve gamma ayarı ister; sonuçlarının açıklanması zordur.",

    "Random Forest":
    "Büyük ormanlar daha fazla kaynak tüketebilir ve boosting modellerinden düşük kalabilir.",
    "XGBoost":
     "Çok sayıda hiperparametresi vardır ve yanlış ayarlanırsa overfitting yapabilir."
}

In [ ]:
model_comparison = pd.DataFrame(
    comparison_rows
)

model_comparison["Güçlü Yön"] = (
    model_comparison["Model"]
    .map(strengths)
)

model_comparison["Zayıf Yön"] = (
    model_comparison["Model"]
    .map(weaknesses)
)

In [ ]:
model_comparison = (
    model_comparison
    .sort_values(
        by="ROC-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

model_comparison.round(4)   

,Model,ROC-AUC,ROC-AUC Std,PR-AUC,F1,Recall,Precision,Accuracy,Ortalama Eğitim Süresi,Güçlü Yön,Zayıf Yön
0,Logistic Regression,0.8461,0.0126,0.6615,0.5924,0.5438,0.6521,0.8019,0.1024,Hızlı ve güçlü bir baseline modeli.,Karmaşık doğrusal olmayan ilişkileri ve özelli...
1,XGBoost,0.8246,0.0068,0.6288,0.5590,0.5151,0.6121,0.7843,0.4923,Önceki ağaçların hatalarını sırayla düzelterek...,Çok sayıda hiperparametresi vardır ve yanlış a...
2,Random Forest,0.8202,0.0120,0.6103,0.5466,0.4849,0.6271,0.7865,1.5572,"Özellik etkileşimlerini öğrenir, scaling istem...",Büyük ormanlar daha fazla kaynak tüketebilir v...
3,SVM-RBF,0.7988,0.0180,0.6374,0.5753,0.4983,0.6811,0.8048,0.5889,RBF kernel sayesinde karmaşık ve eğrisel karar...,Scaling ile dikkatli C ve gamma ayarı ister; s...
4,KNN,0.7821,0.0052,0.5070,0.5466,0.5324,0.5617,0.7657,0.0746,Benzer müşterilerin davranışlarından yararlana...,Scaling ve özellik seçimine duyarlı; yüksek bo...


In [39]:
model_comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

## Scaling Gereksinimi

KNN, müşteriler arasındaki mesafeyi doğrudan kullandığı için ve SVM-RBF de kernel benzerliğini
özellik uzayındaki uzaklıklara göre hesapladığından sayısal
değişkenlerin ölçeklendirilmesi gerekir.

Logistic Regression doğrudan mesafe kullanmasa da ölçeklendirme,
optimizasyonun daha kararlı çalışmasını sağlar.

Random Forest ve XGBoost ise kararlarını özelliklerin belirli eşiklerin
altında veya üstünde olmasına göre veren ağaçlardan oluşur. Ölçeklendirme
değerlerin sırasını değiştirmediği için bu modellerde StandardScaler
kullanılmamıştır.

In [40]:
top_two_models = (
    model_comparison["Model"]
    .head(2)
    .tolist()
)

top_two_models

['Logistic Regression', 'XGBoost']

In [41]:
parameter_spaces = {
    "Logistic Regression": {
        "model__C": np.logspace(
            -3,
            2,
            30
        ),
        "model__penalty": [
            "l1",
            "l2"
        ],
        "model__solver": [
            "liblinear"
        ],
        "model__class_weight": [
            None,
            "balanced"
        ]
    },

    "KNN": {
        "model__n_neighbors": list(
            range(3, 32, 2)
        ),
        "model__weights": [
            "uniform",
            "distance"
        ],
        "model__p": [
            1,
            2
        ]
    },

    "SVM-RBF": {
        "model__C": np.logspace(
            -2,
            2,
            30
        ),
        "model__gamma": np.logspace(
            -4,
            0,
            30
        ),
        "model__class_weight": [
            None,
            "balanced"
        ]
    },

    "Random Forest": {
        "model__n_estimators": [
            200,
            300,
            500,
            700
        ],
        "model__max_depth": [
            None,
            5,
            10,
            15,
            20
        ],
        "model__min_samples_split": [
            2,
            5,
            10,
            20
        ],
        "model__min_samples_leaf": [
            1,
            2,
            4,
            8
        ],
        "model__max_features": [
            "sqrt",
            "log2",
            None
        ],
        "model__class_weight": [
            None,
            "balanced"
        ]
    },

    "XGBoost": {
        "model__n_estimators": [
            100,
            200,
            300,
            500,
            700
        ],
        "model__learning_rate": [
            0.01,
            0.03,
            0.05,
            0.1,
            0.2
        ],
        "model__max_depth": [
            2,
            3,
            4,
            5,
            6
        ],
        "model__min_child_weight": [
            1,
            3,
            5,
            10
        ],
        "model__subsample": [
            0.6,
            0.8,
            1.0
        ],
        "model__colsample_bytree": [
            0.6,
            0.8,
            1.0
        ],
        "model__reg_alpha": [
            0,
            0.01,
            0.1,
            1
        ],
        "model__reg_lambda": [
            0.5,
            1,
            2,
            5
        ]
    }
}

In [42]:
tuning_rows = []
best_estimators = {}

for model_name in top_two_models:
    print(
        f"\nOptimize ediliyor: {model_name}"
    )

    search = RandomizedSearchCV(
        estimator=model_pipelines[model_name],
        param_distributions=(
            parameter_spaces[model_name]
        ),
        n_iter=20,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False
    )

    search.fit(
        X_train,
        y_train
    )

    best_estimators[model_name] = (
        search.best_estimator_
    )

    tuning_rows.append({
        "Model": model_name,
        "En İyi ROC-AUC": (
            search.best_score_
        ),
        "En İyi Parametreler": (
            search.best_params_
        )
    })

    search_results = pd.DataFrame(
        search.cv_results_
    )

    search_results = (
        search_results
        .sort_values("rank_test_score")
    )

    safe_model_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    search_results.to_csv(
        f"../reports/{safe_model_name}_search_results.csv",
        index=False
    )


Optimize ediliyor: Logistic Regression
Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\Emine Bilge\Desktop\telco-churn-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



Optimize ediliyor: XGBoost
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [43]:
tuning_summary = pd.DataFrame(
    tuning_rows
)

tuning_summary

,Model,En İyi ROC-AUC,En İyi Parametreler
0,Logistic Regression,0.846351,"{'model__solver': 'liblinear', 'model__penalty..."
1,XGBoost,0.849783,"{'model__subsample': 1.0, 'model__reg_lambda':..."


In [44]:
tuning_summary.to_csv(
    "../reports/tuning_summary.csv",
    index=False
)

In [45]:
before_tuning = (
    model_comparison[
        ["Model", "ROC-AUC"]
    ]
    .rename(
        columns={
            "ROC-AUC":
            "Optimizasyon Öncesi ROC-AUC"
        }
    )
)

In [46]:
tuning_comparison = tuning_summary.merge(
    before_tuning,
    on="Model",
    how="left"
)

In [47]:
tuning_comparison[
    "ROC-AUC Katkısı"
] = (
    tuning_comparison[
        "En İyi ROC-AUC"
    ]
    - tuning_comparison[
        "Optimizasyon Öncesi ROC-AUC"
    ]
)

In [48]:
tuning_comparison.round(4)

,Model,En İyi ROC-AUC,En İyi Parametreler,Optimizasyon Öncesi ROC-AUC,ROC-AUC Katkısı
0,Logistic Regression,0.8464,"{'model__solver': 'liblinear', 'model__penalty...",0.8461,0.0002
1,XGBoost,0.8498,"{'model__subsample': 1.0, 'model__reg_lambda':...",0.8246,0.0251


In [49]:
tuning_comparison.to_csv(
    "../reports/tuning_comparison.csv",
    index=False
)

## Model Karşılaştırma Sonuçları

ROC-AUC sonucuna göre en başarılı iki model Logistic Regression ve XGBoost olmuştur.

En iyi iki model RandomizedSearchCV kullanılarak optimize edilmiştir.
Her model için 20 farklı hiperparametre konfigürasyonu aynı 5-fold
cross-validation düzeninde denenmiş ve bütün sonuçlar CSV dosyalarına
kaydedilmiştir. Optimizasyon sonucunda Logistic Regression modelinin ROC-AUC değeri
0.0002 yükseldi.XGBoost modelinin ROC-AUC değeri ise 0.0251 yükselmiştir.